In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv("/content/fashion-mnist_train.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df.shape

(60000, 785)

In [ ]:
df.drop(columns=['id'], inplace=True)

KeyError: "['id'] not found in axis"

In [ ]:
df.head()

### Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:,0], test_size=0.2)

### Scaling

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train

In [ ]:
y_train

### Label Encoding

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
y_train

### Numpy arrays to PyTorch tensors

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [ ]:
X_train_tensor.shape

In [ ]:
X_train_tensor

### Defining the model

In [ ]:
class MySimpleNN():

  def __init__(self, X):

    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
    self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self,y_pred, y):
    # Clamp prediction to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss

### Important Parameters

In [ ]:
learning_rate = 0.1
epochs = 25

### Training Pipeline

In [ ]:
# Create model
model = MySimpleNN(X_train_tensor)
# model.weights

In [ ]:
# model.bias

In [ ]:
# define loop
for epoch in range(epochs):
  # forward pass
  y_pred = model.forward(X_train_tensor)
  #print(y_pred)

# loss calsulate
  loss = model.loss_function(y_pred, y_train_tensor)

# backward pass
  loss.backward()

# parameters update
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

# zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

# print loss in each epoch
  print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")

### Evaluation

In [ ]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f"Accuracy: {accuracy.item()}")

## Training Pipeline using Container

### Defining the model

In [ ]:
import torch.nn as nn

class MySimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):

    out = self.linear(features)
    out = self.sigmoid(out)
    return out



In [ ]:
learning_rate = 0.1
epochs = 25

In [ ]:
# define loss function
loss_function = nn.BCELoss()

In [ ]:
# define optimizer
optimizer = torch.optim.SGD(model.parameters(), )

### Training Pipeline

In [ ]:
# Create model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model(X_train_tensor.float())
  #print(y_pred)

# loss calsulate
  loss = loss_function(y_pred, y_train_tensor.view(-1,1).float())

# clear gradients
  optimizer.zero_grad()

# backward pass
  loss.backward()

# parameters update
  optimizer.step()


# print loss in each epoch
  print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")

In [ ]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor.float())
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f"Accuracy: {accuracy.item()}")

## Training With Dataset and DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [ ]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [ ]:
train_dataset[20]

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

### Defining the model

In [ ]:
import torch.nn as nn

class MysimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):

    out = self.linear(features)
    out = self.sigmoid(out)
    return out


In [ ]:
learning_rate = 0.1
epochs = 25

In [ ]:
# create model
model = MySimpleNN(X_train_tensor.shape[1])

# define loss function
loss_function = nn.BCELoss()

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

### Training Pipeline

In [ ]:
# define loop
for epoch in range(epochs):

  for batch_features, batch_labels in train_loader:

    # forward pass
    y_pred = model(batch_features.float())

    # loss calsulate
    loss = loss_function(y_pred, batch_labels.view(-1,1).float())

    # clear gradients
    optimizer.step()

    # parameters update
    optimizer.step()

# print loss in each epoch
  print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")

### Evaluation

In [ ]:
# Model evaluation using test_loader
model.eval()   # Set the model to evalution mode
accuracy_list = []

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    # Forward pass
    y_pred = model(batch_features.float())
    y_pred = (y_pred > 0.5).float   # Convert probabilities to binary predictions

    # Calsulate accuracy for the current batch
    batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
    accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f"Accuracy: {overall_accuracy:.4f}")